# Phase Transformation Foundations

PyTex now has the first stable primitive family for phase-transformation work:

- `OrientationRelationship`
- `TransformationVariant`
- `PhaseTransformationRecord`

This notebook explains the semantics rather than pretending the repo already has a
full parent-reconstruction platform.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalMap,
    CrystalPlane,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    unit_cell = UnitCell(
        lattice=lattice,
        sites=(
            AtomicSite("A1", "Cu", np.array([0.0, 0.0, 0.0])),
            AtomicSite("A2", "Cu", np.array([0.0, 0.5, 0.5])),
            AtomicSite("A3", "Cu", np.array([0.5, 0.0, 0.5])),
            AtomicSite("A4", "Cu", np.array([0.5, 0.5, 0.0])),
        ),
    )
    phase = Phase(
        "fcc_demo",
        lattice=lattice,
        symmetry=symmetry,
        crystal_frame=crystal,
        unit_cell=unit_cell,
    )
    return crystal, specimen, map_frame, detector, lab, phase


In [ ]:
crystal, specimen, map_frame, detector, lab, parent_phase = make_context()
child_crystal = ReferenceFrame(
    "child_crystal",
    FrameDomain.CRYSTAL,
    ("a", "b", "c"),
    Handedness.RIGHT,
)
child_symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=child_crystal)
child_lattice = Lattice(3.2, 3.2, 3.2, 90.0, 90.0, 90.0, crystal_frame=child_crystal)
child_phase = Phase(
    "child_phase",
    lattice=child_lattice,
    symmetry=child_symmetry,
    crystal_frame=child_crystal,
)

relationship = OrientationRelationship(
    name="demo_or",
    parent_phase=parent_phase,
    child_phase=child_phase,
    parent_to_child_rotation=Rotation.from_axis_angle([0.0, 0.0, 1.0], np.pi / 4.0),
)

variants = relationship.generate_variants()
print("Variant count:", len(variants))


In [ ]:
parent_orientation = Orientation(
    rotation=Rotation.identity(),
    crystal_frame=parent_phase.crystal_frame,
    specimen_frame=specimen,
    symmetry=parent_phase.symmetry,
    phase=parent_phase,
)
child_orientations = OrientationSet.from_orientations(
    [
        Orientation(
            rotation=relationship.parent_to_child_rotation,
            crystal_frame=child_phase.crystal_frame,
            specimen_frame=specimen,
            symmetry=child_phase.symmetry,
            phase=child_phase,
        )
    ]
)
record = PhaseTransformationRecord(
    name="demo_record",
    orientation_relationship=relationship,
    parent_orientation=parent_orientation,
    child_orientations=child_orientations,
    variant_indices=np.array([1]),
)
print(record.variant_count)
